<a href="https://colab.research.google.com/github/suyaibalsifat/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/suyaibalsifat/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!git clone https://github.com/SUYAIBALSIFAT/flyrank-ml-internship.git
%cd flyrank-ml-internship

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 157, done.
remote: Counting objects: 100% (157/157), done.
remote: Compressing objects: 100% (113/113), done.
remote: Total 157 (delta 64), reused 94 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (157/157), 1.87 MiB | 5.48 MiB/s, done.
Resolving deltas: 100% (64/64), done.
/content/flyrank-ml-internship


## 1. Ranked actions + reason codes

The playbook queue ranks 3,541 actionable pages (of 5,745 in my held-out test set) by the Week-5 model's predicted probability, filtered to only rows my Week-4 rule also flags with a real reason code — declining_and_low_ctr, declining_with_demand, or ctr_review_candidate. I deliberately excluded model-flagged rows the rule calls "monitor" from the queue itself (19 such rows scored ≥0.90) — see Section 3 for why.

In [2]:
import pandas as pd, numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["needs_review"] = ((df.trend_direction == "down") & (df.impressions_90d >= 100)).astype(int)
feature_cols = ["search_volume","competition","cpc","word_count","char_count",
    "impressions_90d","clicks_90d","sessions_90d","users_90d","engaged_sessions_90d",
    "ai_sessions_90d","scroll_events_90d","days_with_impressions","days_with_sessions",
    "content_age_days","days_since_last_update","ctr","avg_position","engagement_rate",
    "scroll_rate","ai_traffic_pct"]
data = df.dropna(subset=feature_cols + ["client_id"]).copy()
X = data[feature_cols].fillna(0)
y = data["needs_review"]
groups = data["client_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
tr_idx, te_idx = next(gss.split(X, y, groups))
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X.iloc[tr_idx], y.iloc[tr_idx])
data_te = data.iloc[te_idx].copy()
data_te["model_score"] = rf.predict_proba(X.iloc[te_idx])[:, 1]

def reason_code(row):
    declining = (row.trend_direction == "down") and (row.impressions_90d >= 100)
    low_ctr = (row.impressions_90d >= 500) and (0 < row.avg_position <= 20) and (row.ctr < 0.5)
    if declining and low_ctr: return "declining_and_low_ctr"
    elif declining: return "declining_with_demand"
    elif low_ctr: return "ctr_review_candidate"
    return "monitor"

data_te["reason_code"] = data_te.apply(reason_code, axis=1)
data_te["action"] = data_te["reason_code"].map({
    "declining_and_low_ctr": "refresh_review", "declining_with_demand": "refresh_review",
    "ctr_review_candidate": "ctr_review", "monitor": "monitor"})

queue = data_te[data_te.action != "monitor"].sort_values("model_score", ascending=False)
print(f"Actionable queue: {len(queue)} of {len(data_te)} test rows")
queue[["content_id", "model_score", "reason_code", "action", "impressions_90d", "avg_position", "ctr"]].head(10)


Actionable queue: 3541 of 5745 test rows


,content_id,model_score,reason_code,action,impressions_90d,avg_position,ctr
10441,content_f3d5781e6d92,0.955,declining_and_low_ctr,refresh_review,548,7.7,0.0
10482,content_aeb5e26a7f7e,0.955,declining_with_demand,refresh_review,245,6.6,0.0
13333,content_91307f62115f,0.950,declining_with_demand,refresh_review,441,4.3,0.0
4274,content_29e340486da3,0.940,declining_with_demand,refresh_review,173,8.0,0.0
24699,content_8b2adc2bb561,0.940,declining_with_demand,refresh_review,222,8.2,0.0
29017,content_782295e038c2,0.935,declining_with_demand,refresh_review,142,8.6,0.0
18561,content_7a69792f4c29,0.935,declining_with_demand,refresh_review,257,10.3,0.0
24737,content_d86bc0c51b88,0.935,declining_with_demand,refresh_review,414,7.0,0.0
9211,content_ad7989362d4b,0.935,declining_and_low_ctr,refresh_review,664,9.9,0.0
12829,content_f73cc75941bc,0.935,declining_with_demand,refresh_review,278,6.5,0.0


## 2. Intended use and limits

*Who uses this, for what — and where it stoIntended use: a weekly priority list for a content editor with limited review capacity, ranking which pages to look at first.
Who: SEO/content editors, not an automated publishing system.
Where it stops being valid: this queue is scored on a proxy label (currently-declining-with-demand), not a true future outcome — it identifies pages worth a human look, not pages guaranteed to recover if edited. It also reflects the starter dataset's 44 clients only, at one point in time; it isn't validated on a live, ongoing feed.ps being valid.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list

Before acting on ANY row: a human should check the page still exists and is still owned by the client, confirm the "decline" isn't explained by something outside content quality (URL migration, seasonal topic, a merged/redirected page), and sanity-check that ctr = 0.00 rows (there are several in my top scores) aren't a tracking gap rather than a real zero-click page.

What should NOT be automated: no automatic publishing, deletion, or rewriting of pages from this score alone. No automatic client-facing reporting claiming the model "found problems" — it flags candidates, it doesn't diagnose them.

Model/rule disagreement (a real limit I found): 19 pages scored ≥0.90 by the model but were excluded from the queue because my Week-4 rule labels them "monitor" (trend isn't literally "down"). I chose to trust the stricter rule and exclude them rather than silently include model-only picks — a human reviewer should know this cutoff exists and could occasionally look at high-scoring "monitor" pages too.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

Retrain if: the gap between random-split and grouped-split precision@50 (found in Week 6 to be 1.0 vs 0.66) reappears or widens on new data — that gap is the tell that the model is memorizing rather than generalizing.
Retrain if: the base rate of needs_review drifts meaningfully from ~49% (a big shift means client mix or trend patterns changed).
Review the rule itself if: the "declining_with_demand" component keeps looking too close to the label (as found in Week 5) — a future version should use a strictly future-window label instead of a same-window proxy.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

Exporting the queue and a summary metrics file — the paper next week builds directly on these.

In [3]:
import os, json

os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

out_cols = ["content_id", "model_score", "reason_code", "action", "impressions_90d", "avg_position", "ctr"]
queue[out_cols].to_csv("work/outputs/content_action_playbook.csv", index=False)

metrics = {
    "queue_size": int(len(queue)),
    "test_set_size": int(len(data_te)),
    "baseline_precision_at_50": 0.74,
    "model_precision_at_50_grouped_split": 0.66,
    "model_precision_at_50_random_split_leaked": 1.0,
    "base_rate": round(float(y.iloc[te_idx].mean()), 3),
    "high_score_rule_disagreement_count": int(((data_te.model_score >= 0.9) & (data_te.action == "monitor")).sum()),
}
with open("work/outputs/w07_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("Exported queue and metrics.")
print(metrics)

Exported queue and metrics.
{'queue_size': 3541, 'test_set_size': 5745, 'baseline_precision_at_50': 0.74, 'model_precision_at_50_grouped_split': 0.66, 'model_precision_at_50_random_split_leaked': 1.0, 'base_rate': 0.491, 'high_score_rule_disagreement_count': 19}


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.